# Course 21 lab — governed parallel-agent delivery

Northstar must deliver AI-2500 across five repositories without allowing parallel workers to race shared decisions, reuse stale context, or self-approve integration. This notebook uses real temporary Git repositories and a deterministic trusted control plane.

## Objectives and safety boundary

You will compare scheduling strategies, inspect immutable evidence subjects, trigger stale-worker and recovery failures, and assemble an exact-revision integration package. Workers propose; trusted code authenticates, authorizes, executes checks, records evidence, and decides readiness. The lab is offline, credential-free, and does not mutate remote repositories.

## Architecture walkthrough

```text
contract approval ─┬─ document worker ───────────┐
                   ├─ AI worker ────────────────┤
                   └─ review worker ─ API ─ UI ─┤
                                                  ▼
                     exact-revision integration barrier
```

The control plane sits between planning recommendations and consequential repository operations.

In [ ]:
import importlib.util
import json
import sys
from pathlib import Path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from parallel_app.simulation import golden_scenarios, scheduling_comparison
spec = importlib.util.spec_from_file_location('course21_notebook_lab', ROOT / 'lab.py')
course21_lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = course21_lab
spec.loader.exec_module(course21_lab)
report = course21_lab.build_report()
assert report['course_validation']['status'] == 'PASS'

## Baseline experiment — sequential, naive parallel, governed parallel

The same deterministic work durations are scheduled three ways. Elapsed time follows the critical path; it is not the sum of parallel task durations. Naive fan-out adds contract-driven rework.

In [ ]:
comparison = scheduling_comparison(ROOT)
for strategy, values in comparison.items():
    print(strategy, values['elapsed_hours'], 'h elapsed;', values['rework_hours'], 'h rework')
assert comparison['governed_parallel']['elapsed_hours'] < comparison['naive_parallel']['elapsed_hours']

The reference fixture shows that coordination has a cost, but prevents more expensive avoidable rework. This is not a universal benchmark; change graph shape, review capacity, or contract churn and the result may reverse.

## Inspect real evidence subjects

Each repository is initialized, committed twice, tested, and diffed. Full commit IDs—not branch names—identify the baseline and candidate.

In [ ]:
observations = report['repository_observations']
for repository, item in observations.items():
    print(repository, item['candidate_commit'][:12], item['changed_paths'], item['tests_passed'])
assert len(observations) == 5
assert all(len(item['candidate_commit']) == 40 for item in observations.values())

## Failure injection — ownership, fencing, scope, and completion

The conformance suite acquires competing leases, expires and reassigns work, attempts a stale submission, modifies an unauthorized path, and lets a worker claim completion. The expected result comes from control-plane behavior, not a fixture flag.

In [ ]:
golden = golden_scenarios(ROOT)
for case_id in ('PAR-GOLDEN-001', 'PAR-GOLDEN-002', 'PAR-GOLDEN-004', 'PAR-GOLDEN-008'):
    print(case_id, golden[case_id])
assert all(golden[case_id] for case_id in ('PAR-GOLDEN-001', 'PAR-GOLDEN-002', 'PAR-GOLDEN-004', 'PAR-GOLDEN-008'))

Lease expiry alone is insufficient: generation 41 may keep running. Safety appears only when the protected submission service rejects 41 after 42 is current.

## Unknown outcomes and checkpoint recovery

The lab persists logical operation results so a lost PR response reconciles to one external identity. It also verifies a checkpoint's work unit, generation, base commit, contract revision, and current authority before resume.

In [ ]:
for case_id in ('PAR-GOLDEN-006', 'PAR-GOLDEN-007', 'PAR-GOLDEN-011', 'PAR-GOLDEN-012'):
    print(case_id, golden[case_id])
assert golden['PAR-GOLDEN-006'] and golden['PAR-GOLDEN-012']

A restart must not reset fencing generations or forget already-created effects. Retry exhaustion becomes escalation rather than another unbounded agent turn.

## Contract change and selective invalidation

A worker may propose revision 3, but only the contract owner approves it. AI and Review are direct dependents; API and UI are transitively invalidated; Document remains reusable. Old evidence is removed because its contract digest no longer identifies the current subject.

In [ ]:
print('selective contract invalidation', golden['PAR-GOLDEN-005'])
assert golden['PAR-GOLDEN-005']

## Integration barrier

The barrier resolves trusted evidence for five exact commits, the approved contract digest, and six system checks. Substituting a different commit blocks the candidate even if every local pipeline was green.

In [ ]:
assessment = report['candidate_assessment']
print(json.dumps(assessment, indent=2, sort_keys=True))
assert assessment['status'] == 'READY_FOR_INTEGRATION_REVIEW'
assert assessment['merge_authorized'] is False
assert assessment['deployment_authorized'] is False

## Evaluation

The declared population contains 7 positive, 15 negative, 5 boundary, and 3 failure/recovery decisions. Report the numerator and denominator together and keep this teaching population separate from production reliability.

In [ ]:
evaluation = report['evaluation']
print(evaluation['passed'], '/', evaluation['total'], evaluation['population'])
assert (evaluation['passed'], evaluation['total']) == (30, 30)
assert report['system_scenarios']['passed'] == 6

## Production upgrade

| Fixture mechanism | Production upgrade |
| --- | --- |
| in-process lock | transactional store or service with tested fencing semantics |
| configured worker registry | attested workload identity and short-lived scoped credentials |
| local Git repositories | protected mutation API, pinned PR heads, signed provenance |
| JSON snapshot | durable versioned state, queue redelivery, failover and recovery tests |
| synthetic durations | representative traces segmented by risk and complexity |
| integration eligibility | separate human/organizational merge and release gates |

## Exercises

1. Add bounded lease renewal and test the expiry boundary.
2. Add an attempt ID without changing logical operation identity.
3. Inject an incomplete dependency graph and design the review obligation.
4. Add reviewer capacity to the event simulation.
5. Explain why a green `(AI-A, Review-B)` receipt cannot prove `(AI-A, Review-C)`.
6. Design the owner response to revision 3 while three workers are active.

## Summary

Parallelism is safe only when work is independently executable, ownership is fenced at the protected resource, retries reconcile durable outcomes, resume validates current context, contract changes invalidate affected evidence, and integration binds the exact tested revision set. The reference package is reviewable—not self-authorized.